# 01 — SDK Basics

This notebook covers the core FactPy SDK surface:
- Entity / Field / Identity schema definitions
- Schema preflight validation
- SDKStore initialisation (in-memory and file-backed)
- Batch writes with `preview` and `commit`
- Low-level writes: `ref`, `set`, `add`, `retract`
- Read APIs: `get`, `find`, `EntitySnapshot`
- Edit API: `sdk.edit(...)` with auto-commit
- Ingest API for external normalised data
- Common boundary checks (expected errors)

**Prerequisites:** None.  
**Next:** [02_rules_and_derivations.ipynb](02_rules_and_derivations.ipynb)

## 0. Imports

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path(".").resolve().parent / "src"))

In [2]:
from __future__ import annotations

import os, shutil, tempfile, warnings
from pathlib import Path
from uuid import uuid4
from pprint import pprint

from factpy_kernel.sdk import (
    SDKStore, Entity, Identity, Field,
    schema_preflight_from_classes,
    SDKStoreError, SDKSchemaError,
    CardinalityError, EntityNotFoundError, EditorClosedError,
)

## 1. Schema Definitions

Covers `Identity(primary_key, default_factory)`, `Field(cardinality)`, and entity-reference fields.

In [3]:
class Country(Entity):
    iso_code: str = Identity(primary_key=True)
    name: str = Field(cardinality="single")


class Language(Entity):
    code: str = Identity(primary_key=True)
    name: str = Field(cardinality="single")


class User(Entity):
    user_id: str = Identity(primary_key=True)
    locale: str = Identity()
    name: str = Field(cardinality="single")
    aliases: str = Field(cardinality="multi")
    age: int = Field(cardinality="single")
    country: Country = Field(cardinality="single")
    tag: str = Field(cardinality="multi")


class LivesIn(Entity):
    uid: str = Identity(primary_key=True, default_factory="uuid4")
    user: User = Field(cardinality="single")
    country: Country = Field(cardinality="single")
    since: int = Field(cardinality="single")


class HasLanguage(Entity):
    uid: str = Identity(primary_key=True, default_factory="uuid4")
    country: Country = Field(cardinality="single")
    language: Language = Field(cardinality="single")


class Speaks(Entity):
    uid: str = Identity(primary_key=True, default_factory="uuid4")
    user: User = Field(cardinality="single")
    language: Language = Field(cardinality="single")

## 2. Schema Preflight

Run health checks early in CI or at startup.

In [4]:
preflight = schema_preflight_from_classes([Country, Language, User, LivesIn, HasLanguage, Speaks])
print("preflight ok:", preflight["ok"])
pprint(preflight.get("summary", {}))

preflight ok: True
{'entity_count': 6,
 'pred_ids': ['Country:exists',
              'HasLanguage:exists',
              'Language:exists',
              'LivesIn:exists',
              'Speaks:exists',
              'User:exists',
              'country:iso_code',
              'country:name',
              'has_language:country',
              'has_language:language',
              'has_language:uid',
              'language:code',
              'language:name',
              'lives_in:country',
              'lives_in:since',
              'lives_in:uid',
              'lives_in:user',
              'speaks:language',
              'speaks:uid',
              'speaks:user',
              'user:age',
              'user:aliases',
              'user:country',
              'user:locale',
              'user:name',
              'user:tag',
              'user:user_id'],
 'predicate_count': 27}


## 3. Store Initialisation

Two patterns: in-memory and file-backed ledger.

In [5]:
classes = [Country, Language, User, LivesIn, HasLanguage, Speaks]

# In-memory
sdk = SDKStore.from_schema_classes(classes, default_row_format="dict")

# File-backed ledger
ledger_path = Path(tempfile.gettempdir()) / f"factpy_example_{uuid4().hex}.db"
sdk_file = SDKStore.from_schema_classes(classes, ledger_path=str(ledger_path))

print("in-memory sdk:", type(sdk).__name__)
print("file ledger:", ledger_path)

in-memory sdk: SDKStore
file ledger: /var/folders/05/6btr2vg13b9gvgs3gxt8fw_40000gn/T/factpy_example_5378301fccb6407d8f939051dd51630d.db


## 4. Batch Writes

`sdk.batch()` supports `preview`, partial identity with `bind`, and explicit `commit`.
The context manager does **not** auto-commit on exit.

In [6]:
tx = sdk.batch(meta={"trace_id": "seed-001", "source": "demo"})

# Countries
de = tx.entity(Country, iso_code="DE")
de.name.set("Germany")
cn = tx.entity(Country, iso_code="CN")
cn.name.set("China")

# Languages
de_lang = tx.entity(Language, code="de"); de_lang.name.set("German")
zh_lang = tx.entity(Language, code="zh"); zh_lang.name.set("Chinese")
en_lang = tx.entity(Language, code="en"); en_lang.name.set("English")

# User with full identity
u1 = tx.entity(User, user_id="u-001", locale="zh")
u1.name.set("Alice"); u1.aliases.add("Alicia"); u1.age.set(30)
u1.country.set(de); u1.tag.add("vip")

# User with partial identity → bind
u2 = tx.entity(User, user_id="u-002")
u2 = u2.bind(locale="en")
u2.name.set("Bob"); u2.aliases.add("Bobby"); u2.age.set(28)
u2.country.set(cn); u2.tag.add("staff")

# Relationships
li1 = tx.entity(LivesIn); li1.user.set(u1); li1.country.set(de); li1.since.set(2019)
li2 = tx.entity(LivesIn); li2.user.set(u2); li2.country.set(cn); li2.since.set(2021)
hl1 = tx.entity(HasLanguage); hl1.country.set(de); hl1.language.set(de_lang)
hl2 = tx.entity(HasLanguage); hl2.country.set(cn); hl2.language.set(zh_lang)

plan = tx.preview()
print("planned ops:", len(plan.ops))

seed_result = tx.commit()
print("written assertions:", len(seed_result.apply_result.assertion_ids))

planned ops: 47
written assertions: 49


In [7]:
# Context manager does NOT auto-commit
with sdk.batch() as tx_no_commit:
    ghost = tx_no_commit.entity(User, user_id="u-no-commit", locale="zh")
    ghost.name.set("WillNotPersist")

print("no-commit user exists?", sdk.get(User, user_id="u-no-commit", locale="zh") is not None)

no-commit user exists? False


### 4.1 Identity Immutability

In [8]:
with sdk.batch() as tx_err:
    incomplete = tx_err.entity(User, user_id="u-003")
    try:
        incomplete.name.set("Charlie")
    except SDKStoreError as exc:
        print("incomplete identity write blocked:", exc)

    incomplete = incomplete.bind(locale="fr")
    incomplete.name.set("Charlie")
    tx_err.commit()

with sdk.edit(User, user_id="u-001", locale="zh") as editor:
    try:
        editor.locale.set("en")
    except SDKStoreError as exc:
        print("identity immutable:", exc)

incomplete identity write blocked: User#1.name: identity is incomplete for User; missing: ['locale']. Bind missing identity via handle.bind(...).
identity immutable: identity field 'locale' is immutable in editor; open a new editor with different identity instead


## 5. Low-Level Writes: `ref / set / add / retract`

In [9]:
u1_ref = sdk.ref(User, user_id="u-001", locale="zh")

asrt_alias = sdk.add(User.aliases, u1_ref, "Alice Cooper",
                      meta={"source": "low-level", "trace_id": "ll-1"})
asrt_age = sdk.set(User.age, u1_ref, 31,
                    meta={"source": "low-level", "trace_id": "ll-2",
                          "version": "v2", "valid_from": "2024-01-01T00:00:00+00:00"})
revoker = sdk.retract(asrt_alias, meta={"source": "low-level", "trace_id": "ll-3"})

print("alias asrt:", asrt_alias)
print("age asrt:", asrt_age)
print("revoker:", revoker)

alias asrt: 6578b8775ddb461a970ad53827ec3b3a
age asrt: e9f5ec195d664ce39d26a21687dba4f8
revoker: 7b076dfa3d3f41d1863b4f4bf930b3c8


## 6. Read APIs: `get`, `find`, `EntitySnapshot`

In [10]:
snap = sdk.get(User, user_id="u-001", locale="zh")
print("ref:", snap.ref)
print("name:", snap.name)
print("aliases:", snap.aliases)
print("age:", snap.age)
print("country:", snap.country)

rows_vip = sdk.find(User, tag="vip", limit=20)
print("vip refs:", [r.ref for r in rows_vip])

ref: idref_v1:User:mpdat3awltdenidj64aocoihw44q7n4m5tzu3mmnld5wimwtc7ba
name: Alice
aliases: ('Alicia',)
age: 31
country: idref_v1:Country:6yld57v2sjmk2up44c6xufn6gul22a5gclpagrqsx3rx53fk6qtq
vip refs: ['idref_v1:User:mpdat3awltdenidj64aocoihw44q7n4m5tzu3mmnld5wimwtc7ba']


In [11]:
# EntitySnapshot assertion namespace
snap = sdk.get(User, user_id="u-001", locale="zh")
print("age.active count:", len(snap.assertions.age.active))
print("age.history count:", len(snap.assertions.age.history))
print("age.version('v2'):", [r.value for r in snap.assertions.age.version("v2")])

age.active count: 2
age.history count: 2
age.version('v2'): [31]


## 7. Edit API

In [12]:
with sdk.edit(User, user_id="u-001", locale="zh") as editor:
    editor.aliases.add("Alice Z")
    editor.age.set(32)

# Cardinality violation
try:
    with sdk.edit(User, user_id="u-001", locale="zh") as editor:
        editor.age.add(999)
except CardinalityError as exc:
    print("cardinality blocked:", exc)

# Closed editor
ed = sdk.edit(User, user_id="u-001", locale="zh")
ed.rollback()
try:
    ed.age.set(1)
except EditorClosedError as exc:
    print("closed editor blocked:", exc)

cardinality blocked: field 'age' has cardinality=single; .add is only valid for multi
closed editor blocked: editor is closed


## 8. Ingest API

In [13]:
snap_before = sdk.get(User, user_id="u-001", locale="zh")
old_alias_id = (snap_before.assertions.aliases.active[0].asrt_id
                if snap_before.assertions.aliases.active else None)

items = [
    {"kind": "add", "field": User.tag, "e_ref": u1_ref, "value": "priority"},
    {"kind": "set", "field": User.age, "e_ref": u1_ref, "value": 33,
     "meta": {"version": "ingest-v3", "valid_from": "2025-01-01T00:00:00+00:00"}},
]
if old_alias_id:
    items.append({"kind": "retract", "asrt_id": old_alias_id})

result = sdk.ingest(items, meta={"source": "ingest-demo", "trace_id": "ing-001"})
print("written:", len(result.written_assertion_ids))
print("duplicates:", result.duplicate_count, "skipped:", result.skipped_count)

written: 3
duplicates: 0 skipped: 0


## 9. Boundary Checks (Expected Errors)

In [14]:
try:
    sdk.run("select * from ...")
except SDKStoreError as exc:
    print("string rule rejected:", exc)

try:
    _ = sdk.get(User, user_id="u-001")  # missing locale
except SDKSchemaError as exc:
    print("missing identity:", exc)

try:
    sdk.edit(User, user_id="u-not-found", locale="zh")
except EntityNotFoundError as exc:
    print("missing entity:", exc)

string rule rejected: string rule DSL is not supported in SDK v1; use Rule object, RuleSpec, or structured rule dict
missing identity: missing identity fields for get(User): ['locale']
missing entity: entity not found: User with identity {'user_id': 'u-not-found', 'locale': 'zh'}


---
**Next:** [02_rules_and_derivations.ipynb](02_rules_and_derivations.ipynb) — Rule DSL, Derivation, evaluate, and accept